In [19]:
import numpy as np
import sklearn
import pandas as pd
import matplotlib
import seaborn as sns

# 각 라이브러리 버전 출력
print("Numpy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)
print("Pandas version:", pd.__version__)
print("Matplotlib version:", matplotlib.__version__)
print("Seaborn version:", sns.__version__)

Numpy version: 1.26.4
Scikit-learn version: 1.4.2
Pandas version: 2.2.2
Matplotlib version: 3.8.4
Seaborn version: 0.13.2


In [20]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import  OrdinalEncoder
from sklearn.ensemble import ExtraTreesClassifier

In [21]:
import sys
import os
import pandas as pd

# 현재 작업 디렉토리 경로를 가져와 shared codes 폴더의 위치를 sys.path에 추가합니다.
# sys.path에 추가된 경로에 있는 py 폴더는 임포트할 수 있다.
current_dir = os.getcwd()
shared_codes_dir = os.path.join(current_dir, '../shared codes')
sys.path.append(shared_codes_dir)


# cover_nan 모듈을 임포트
from cover_nan_0215_dahun import missing_value_removal_function

# 원본 train 데이터 로드
train = pd.read_csv("../shared codes/data/train.csv")
test = pd.read_csv("../shared codes/data/test.csv")

# missing_value_removal_function 사용
train_young, train_middle, train_old, train_unknown = missing_value_removal_function(train)
test_young, test_middle, test_old, test_unknown = missing_value_removal_function(test)

✅ '대리모 여부' 결측값을 최빈값 (0.0) 으로 대체 완료!
✅ 컬럼 삭제 완료: ['PGD 시술 여부', 'PGS 시술 여부', '난자 해동 경과일', '배아 해동 경과일']
✅ '난자 채취 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '난자 혼합 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '배아 이식 경과일' 결측값을 중앙값 (3.0) 으로 대체 완료!
✅ '대리모 여부' 결측값을 최빈값 (0.0) 으로 대체 완료!
✅ 컬럼 삭제 완료: ['PGD 시술 여부', 'PGS 시술 여부', '난자 해동 경과일', '배아 해동 경과일']
✅ '난자 채취 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '난자 혼합 경과일' 결측값을 중앙값 (0.0) 으로 대체 완료!
✅ '배아 이식 경과일' 결측값을 중앙값 (3.0) 으로 대체 완료!


In [22]:
def data_preprocessing(train, test):
    # 미리 ID 저장

    index_test = test['ID'].copy() 

    # Drop ID and target columns
    X = train.drop(['임신 성공 여부', 'ID'], axis=1)
    y = train['임신 성공 여부']
    test = test.drop('ID', axis=1)

    # Categorical columns
    categorical_columns = X.select_dtypes(include=['object', 'category']).columns.tolist()

    # Encoding
    from sklearn.preprocessing import OrdinalEncoder
    ordinal_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)

    X_train_encoded = X.copy()
    X_train_encoded[categorical_columns] = ordinal_encoder.fit_transform(X[categorical_columns])

    X_test_encoded = test.copy()
    X_test_encoded[categorical_columns] = ordinal_encoder.transform(test[categorical_columns])

    # Drop unnecessary columns
    columns_to_drop = [
        "남성 주 불임 원인",
        "남성 부 불임 원인",
        "불임 원인 - 정자 농도",
        "불임 원인 - 정자 면역학적 요인",
        "불임 원인 - 정자 운동성",
        "불임 원인 - 정자 형태",
        '배란 유도 유형'
    ]
    X_train_encoded = X_train_encoded.drop(columns=columns_to_drop)    
    X_test_encoded = X_test_encoded.drop(columns=columns_to_drop)

    # ID 다시 추가

    X_test_encoded['ID'] = index_test.values

    return X_train_encoded, X_test_encoded, y


In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 🔹 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'  # Windows (맑은 고딕)
plt.rcParams['axes.unicode_minus'] = False  # 마이너스 깨짐 방지

# 🔹 Feature Importance Analysis Function

def feature_importance_analysis(X_train, y_train, top_n=20):
    """
    RandomForest, XGBoost, LightGBM, CatBoost 모델의 변수 중요도를 분석하고 시각화합니다.
    """
    # 1️⃣ 랜덤 포레스트 모델 학습
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)

    # 2️⃣ XGBoost 모델 학습
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42)
    xgb_model.fit(X_train, y_train)

    # 3️⃣ LightGBM 모델 학습
    lgb_model = lgb.LGBMRegressor(n_estimators=100, random_state=42)
    lgb_model.fit(X_train, y_train)

    # 4️⃣ CatBoost 모델 학습
    cat_model = CatBoostRegressor(iterations=100, verbose=0, random_seed=42)
    cat_model.fit(X_train, y_train)

    # 🔹 변수 중요도 추출
    rf_importances = rf_model.feature_importances_
    xgb_importances = xgb_model.feature_importances_
    lgb_importances = lgb_model.feature_importances_
    cat_importances = cat_model.get_feature_importance()

    feature_names = X_train.columns

    # 🔹 중요도 데이터프레임 생성
    importance_df = pd.DataFrame({
        'Feature': feature_names,
        'RandomForest': rf_importances,
        'XGBoost': xgb_importances,
        'LightGBM': lgb_importances,
        'CatBoost': cat_importances
    })

    # 🔹 상위 N개만 정렬하여 가져오기
    importance_df['Mean Importance'] = importance_df[['RandomForest', 'XGBoost', 'LightGBM', 'CatBoost']].mean(axis=1)
    importance_df = importance_df.sort_values(by='Mean Importance', ascending=False).reset_index(drop=True)

    return importance_df.head(top_n)


In [24]:
# 🔹 데이터 전처리 및 병합
X_train_encoded_young, X_test_encoded_young, y_young = data_preprocessing(train_young, test_young)
X_train_encoded_middle, X_test_encoded_middle, y_middle = data_preprocessing(train_middle, test_middle)
X_train_encoded_old, X_test_encoded_old, y_old = data_preprocessing(train_old, test_old)
X_train_encoded_unknown, X_test_encoded_unknown, y_unknown = data_preprocessing(train_unknown, test_unknown)

# 🔹 데이터 병합
X_train_combined = pd.concat([X_train_encoded_young, X_train_encoded_middle, X_train_encoded_old, X_train_encoded_unknown], axis=0)
y_combined = pd.concat([y_young, y_middle, y_old, y_unknown], axis=0)

# 🔹 Feature Importance Analysis
important_features_df = feature_importance_analysis(X_train_combined, y_combined)
important_features = important_features_df['Feature'].tolist()
print("🔹 중요한 변수들:", important_features)

# 🔹 Important Features만 선택하여 Train과 Test에 적용

X_train_encoded_young = X_train_encoded_young[important_features]
X_train_encoded_middle = X_train_encoded_middle[important_features]
X_train_encoded_old = X_train_encoded_old[important_features]
X_train_encoded_unknown = X_train_encoded_unknown[important_features]

important_features.append('ID')

X_test_encoded_young = X_test_encoded_young[important_features]
X_test_encoded_middle = X_test_encoded_middle[important_features]
X_test_encoded_old = X_test_encoded_old[important_features]
X_test_encoded_unknown = X_test_encoded_unknown[important_features]

[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012204 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 636
[LightGBM] [Info] Number of data points in the train set: 256351, number of used features: 47
[LightGBM] [Info] Start training from score 0.258349
🔹 중요한 변수들: ['이식된 배아 수', '수집된 신선 난자 수', '특정 시술 유형', '배아 이식 경과일', '총 생성 배아 수', '시술 시기 코드', '저장된 배아 수', 'IVF 시술 횟수', '클리닉 내 총 시술 횟수', '혼합된 난자 수', '미세주입된 난자 수', '미세주입에서 생성된 배아 수', '파트너 정자와 혼합된 난자 수', '해동된 배아 수', '불임 원인 - 남성 요인', '미세주입 후 저장된 배아 수', '총 시술 횟수', '총 임신 횟수', '불임 원인 - 배란 장애', '총 출산 횟수']


In [25]:
important_features

['이식된 배아 수',
 '수집된 신선 난자 수',
 '특정 시술 유형',
 '배아 이식 경과일',
 '총 생성 배아 수',
 '시술 시기 코드',
 '저장된 배아 수',
 'IVF 시술 횟수',
 '클리닉 내 총 시술 횟수',
 '혼합된 난자 수',
 '미세주입된 난자 수',
 '미세주입에서 생성된 배아 수',
 '파트너 정자와 혼합된 난자 수',
 '해동된 배아 수',
 '불임 원인 - 남성 요인',
 '미세주입 후 저장된 배아 수',
 '총 시술 횟수',
 '총 임신 횟수',
 '불임 원인 - 배란 장애',
 '총 출산 횟수',
 'ID']

In [26]:
X_train_encoded_young = pd.concat([X_train_encoded_young, y_young], axis=1)
X_train_encoded_middle = pd.concat([X_train_encoded_middle, y_middle], axis=1)
X_train_encoded_old = pd.concat([X_train_encoded_old, y_old], axis=1)
X_train_encoded_unknown = pd.concat([X_train_encoded_unknown, y_unknown], axis=1)

test_young_id = X_test_encoded_young["ID"]
X_test_encoded_young= X_test_encoded_young.drop(columns=["ID"])

test_middle_id = X_test_encoded_middle["ID"]
X_test_encoded_middle= X_test_encoded_middle.drop(columns=["ID"])

test_old_id = X_test_encoded_old["ID"]
X_test_encoded_old= X_test_encoded_old.drop(columns=["ID"])

test_unknown_id = X_test_encoded_unknown["ID"]
X_test_encoded_unknown= X_test_encoded_unknown.drop(columns=["ID"])

In [ ]:
import pandas as pd
from autogluon.tabular import TabularPredictor

def train_model(training_data, target_variable, config):
    """
    주어진 데이터와 하이퍼파라미터 설정을 사용하여 모델을 학습하고,
    학습된 TabularPredictor 객체를 반환합니다.
    """
    model = TabularPredictor(label=target_variable, eval_metric="roc_auc")
    model.fit(
        training_data,
        presets="best_quality",
        num_bag_folds=10,
        hyperparameters=config,
        num_stack_levels=1
    )
    return model


def save_submission(model, test_data, id_series, output_path):
    """
    테스트 데이터에 대해 확률 예측을 수행하고, 제출 파일을 생성하여 저장합니다.
    """
    # 예측 수행 (양성 클래스의 확률 사용)
    prediction_probs = model.predict_proba(test_data)
    submission_data = pd.DataFrame({
        "ID": id_series,
        "probability": prediction_probs[1]
    })
    return submission_data


def combine_submissions(submission_young, submission_middle, submission_old, submission_unknown, output_path="./Result/Submission_combined.csv"):
    """
    세 개의 제출 파일을 합치고 저장합니다.
    """
    combined_submission = pd.concat([submission_young, submission_middle, submission_old, submission_unknown]).reset_index(drop=True)
    combined_submission.to_csv(output_path, index=False)
    print(f"Combined submission 생성 : {output_path}")


def main():
    # 하이퍼파라미터 설정 (각 모델에 대해 기본 설정)
    hyperparams = {
        "GBM": {},
        "CAT": {},
        "XGB": {}
    }
    
    target_col = "임신 성공 여부"
    
    # Young 모델 학습 및 예측
    trained_model_young = train_model(X_train_encoded_young, target_col, hyperparams)
    submission_young = save_submission(trained_model_young, X_test_encoded_young, test_young_id, output_path="./Result/Submission_young.csv")
    
    # Middle 모델 학습 및 예측
    trained_model_middle = train_model(X_train_encoded_middle, target_col, hyperparams)
    submission_middle = save_submission(trained_model_middle, X_test_encoded_middle, test_middle_id, output_path="./Result/Submission_middle.csv")
    
    # Old 모델 학습 및 예측
    trained_model_old = train_model(X_train_encoded_old, target_col, hyperparams)
    submission_old = save_submission(trained_model_old, X_test_encoded_old, test_old_id, output_path="./Result/Submission_old.csv")

    # Old 모델 학습 및 예측
    trained_model_unknown = train_model(X_train_encoded_unknown, target_col, hyperparams)
    submission_unknown = save_submission(trained_model_unknown, X_test_encoded_unknown, test_unknown_id, output_path="./Result/Submission_unknown.csv")

    # 세 개의 제출 파일 합치기
    combine_submissions(submission_young, submission_middle, submission_old, submission_unknown)


if __name__ == "__main__":
    main()


No path specified. Models will be saved in: "AutogluonModels\ag-20250222_130144"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.2
Python Version:     3.12.3
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.22621
CPU Count:          16
Memory Avail:       16.87 GB / 31.93 GB (52.8%)
Disk Space Avail:   196.11 GB / 930.76 GB (21.1%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=10, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the data. Then holdou

Submission 생성 : ./Result/Submission_young.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0  WeightedEnsemble_L2       0.704059   0.704401     roc_auc        2.510323       0.730475  27.361709                 0.010005                0.008004           0.696522            2       True          4
1  WeightedEnsemble_L3       0.704059   0.704401     roc_auc        2.511317       0.731470  27.962556                 0.010999                0.008998           1.297369            3       True          8
2      CatBoost_BAG_L1       0.703892   0.703838     roc_auc        0.607471       0.032017  17.440262                 0.607471                0.032017          17.440262            1       True          2
3       XGBoost_BAG_L1       0.703616   0.703252     roc_auc        0.388930       0.270653   4.470983                 0.388930          

Submission 생성 : ./Result/Submission_middle.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      CatBoost_BAG_L1       0.727957   0.739512     roc_auc        0.594102       0.020524   9.251181                 0.594102                0.020524           9.251181            1       True          2
1  WeightedEnsemble_L3       0.727586   0.739803     roc_auc        0.939450       0.155072  13.037673                 0.007999                0.004506           0.718039            3       True          8
2  WeightedEnsemble_L2       0.727586   0.739803     roc_auc        0.940451       0.155566  12.684797                 0.009000                0.005000           0.365164            2       True          4
3      CatBoost_BAG_L2       0.726502   0.737084     roc_auc        2.539955       0.302012  24.454241                 0.108036          

Submission 생성 : ./Result/Submission_old.csv


Leaderboard on holdout data (DyStack):
                 model  score_holdout  score_val eval_metric  pred_time_test  pred_time_val   fit_time  pred_time_test_marginal  pred_time_val_marginal  fit_time_marginal  stack_level  can_infer  fit_order
0      LightGBM_BAG_L2       0.834244   0.872468     roc_auc        2.633578       0.051037  10.990206                 0.089028                0.007013           2.917069            2       True          5
1       XGBoost_BAG_L2       0.821494   0.868132     roc_auc        2.789622       0.065034  10.301798                 0.245072                0.021010           2.228661            2       True          7
2  WeightedEnsemble_L3       0.821494   0.880898     roc_auc        2.983197       0.082037  17.477338                 0.009512                0.000000           0.022012            3       True          8
3      LightGBM_BAG_L1       0.797814   0.848272     roc_auc        1.619624       0.010005   2.767342                 1.619624          

Submission 생성 : ./Result/Submission_unknown.csv
Combined submission 생성 : ./Result/Submission_combined.csv


In [28]:
X_train_encoded_young.info()

<class 'pandas.core.frame.DataFrame'>
Index: 113727 entries, 0 to 256346
Data columns (total 21 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   이식된 배아 수          113727 non-null  float64
 1   수집된 신선 난자 수       113727 non-null  float64
 2   특정 시술 유형          113727 non-null  float64
 3   배아 이식 경과일         113727 non-null  float64
 4   총 생성 배아 수         113727 non-null  float64
 5   시술 시기 코드          113727 non-null  float64
 6   저장된 배아 수          113727 non-null  float64
 7   IVF 시술 횟수         113727 non-null  float64
 8   클리닉 내 총 시술 횟수     113727 non-null  float64
 9   혼합된 난자 수          113727 non-null  float64
 10  미세주입된 난자 수        113727 non-null  float64
 11  미세주입에서 생성된 배아 수   113727 non-null  float64
 12  파트너 정자와 혼합된 난자 수  113727 non-null  float64
 13  해동된 배아 수          113727 non-null  float64
 14  불임 원인 - 남성 요인     113727 non-null  int64  
 15  미세주입 후 저장된 배아 수   113727 non-null  float64
 16  총 시술 횟수           113727 